In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

### Read Silver Tables

In [0]:
matches_df = spark.table("ipl_analytics.silver.matches")
players_df = spark.table("ipl_analytics.silver.players")
teams_df = spark.table("ipl_analytics.silver.teams")
deliveries_df = spark.table("ipl_analytics.silver.deliveries")

In [0]:
display(matches_df)

display(players_df)

display(teams_df)

display(deliveries_df)

### Check Record Counts

In [0]:
print("Matches :", matches_df.count())

print("Players :", players_df.count())

print("Teams :", teams_df.count())

print("Deliveries :", deliveries_df.count())

### Create dim_player

In [0]:
from pyspark.sql.functions import monotonically_increasing_id,col

dim_player = (
    players_df
    .select(
        col("player_id").alias("player_key"),
        "player_id",
        "player_name",
        "role",
        "team",
        "nationality"
    )
    .dropDuplicates()
)



### (Write Gold Table)

In [0]:
(
    dim_player.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ipl_analytics.gold.dim_player")
)

### Verification

In [0]:
display(spark.table("ipl_analytics.gold.dim_player"))

In [0]:
%sql
SELECT COUNT(*) FROM ipl_analytics.gold.dim_player;

### Create dim_team

In [0]:

dim_team = (
    teams_df
    .select(
        col("team_id").alias("team_key"),
        "team_id",
        "team_name",
        "coach",
        "captain",
        "home_ground"
    )
    .dropDuplicates()
)


display(dim_team)

### Write Gold Table

In [0]:
(dim_team.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ipl_analytics.gold.dim_team"))

### Verify

In [0]:
display(spark.table("ipl_analytics.gold.dim_team"))

In [0]:
%sql
SELECT COUNT(*) FROM ipl_analytics.gold.dim_team;

### Create dim_venue

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

dim_venue = (
    matches_df
    .select("venue")
    .distinct()
    .withColumn("venue_key", monotonically_increasing_id())
    .select(
        "venue_key",
        "venue"
    )
)

display(dim_venue)

### Write Gold Table

In [0]:
(dim_venue.write
.format("delta")
.mode("overwrite")
.option("overwriteSchema", "true")
.saveAsTable("ipl_analytics.gold.dim_venue"))

### Verify

In [0]:
display(spark.table("ipl_analytics.gold.dim_venue"))

In [0]:
%sql
SELECT COUNT(*) FROM ipl_analytics.gold.dim_venue;

### Create fact_match_summary

In [0]:
fact_match_summary = matches_df.alias("m") \
.join(
    dim_venue.alias("v"),
    col("m.venue")==col("v.venue"),
    "left"
) \
.select(
    col("m.match_id"),
    col("v.venue_key"),
    col("m.team1"),
    col("m.team2"),
    col("m.winner"),
    col("m.toss_winner"),
    col("m.match_date"),
    col("m.season"),
    col("m.venue")
)

display(fact_match_summary)

### Write Gold Table

In [0]:
(
    fact_match_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ipl_analytics.gold.fact_match_summary")
)

### Verification

In [0]:
display(spark.table("ipl_analytics.gold.fact_match_summary"))

In [0]:
%sql
SELECT COUNT(*) FROM ipl_analytics.gold.fact_match_summary;

### Create fact_player_performance

In [0]:
from pyspark.sql.functions import *

fact_player_performance = (
    deliveries_df
    .join(
        dim_player,
        deliveries_df.batsman == dim_player.player_name,
        "left"
    )
    .groupBy(
        "player_key",
        "match_id"
    )
    .agg(
        sum("runs_off_bat").alias("total_runs"),
        count("*").alias("balls_faced"),
        sum("extras").alias("extras"),
        sum(when(col("runs_off_bat") == 4, 1).otherwise(0)).alias("fours"),
        sum(when(col("runs_off_bat") == 6, 1).otherwise(0)).alias("sixes"),
        sum("is_wicket").alias("dismissals")
    )
)

### Write Gold Table

In [0]:
(fact_player_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ipl_analytics.gold.fact_player_performance"))

### Verification

In [0]:
display(spark.table("ipl_analytics.gold.fact_player_performance"))

In [0]:
%sql
SELECT COUNT(*) FROM ipl_analytics.gold.fact_player_performance;

### Verify All Gold Tables

In [0]:
print("dim_player :", spark.table("ipl_analytics.gold.dim_player").count())

print("dim_team :", spark.table("ipl_analytics.gold.dim_team").count())

print("dim_venue :", spark.table("ipl_analytics.gold.dim_venue").count())

print("fact_match_summary :", spark.table("ipl_analytics.gold.fact_match_summary").count())

print("fact_player_performance :", spark.table("ipl_analytics.gold.fact_player_performance").count())

In [0]:
display(deliveries_df.select("batsman").distinct())

## --------------------------------------------------------------------------------

### Create SCD Player Dimension

### Verify

### Step 9 – Verify All Gold Tables

In [0]:
%sql
SELECT
    winner AS team,
    COUNT(*) AS matches_won
FROM ipl_analytics.gold.fact_match_summary
GROUP BY winner
ORDER BY matches_won DESC;

###Business Query 3 – Matches Played at Each Venue

In [0]:
%sql
SELECT
    v.venue,
    COUNT(*) AS total_matches
FROM ipl_analytics.gold.dim_venue v
JOIN ipl_analytics.gold.fact_match_summary m
ON v.venue = m.venue
GROUP BY v.venue
ORDER BY total_matches DESC;

In [0]:
%sql
SELECT
    role,
    COUNT(*) AS players
FROM ipl_analytics.gold.dim_player
GROUP BY role;

In [0]:
%sql
SELECT
    team,
    COUNT(*) AS total_players
FROM ipl_analytics.gold.dim_player
GROUP BY team
ORDER BY total_players DESC;

In [0]:
%sql
SELECT
    toss_winner,
    winner,
    COUNT(*) AS matches
FROM ipl_analytics.gold.fact_match_summary
GROUP BY toss_winner, winner
ORDER BY matches DESC;

In [0]:
%sql 
SELECT 
    p.player_name AS batsman, 
    SUM(f.balls_faced) AS balls 
FROM ipl_analytics.gold.fact_player_performance f
JOIN ipl_analytics.gold.dim_player p 
  ON f.player_key = p.player_key
GROUP BY p.player_name 
ORDER BY balls DESC 
LIMIT 10;

In [0]:
spark.table("ipl_analytics.gold.fact_player_performance").printSchema()

In [0]:
dim_player = spark.table("ipl_analytics.gold.dim_player")

In [0]:
deliveries = spark.table("ipl_analytics.silver.ball_by_ball_clean")

In [0]:
deliveries = spark.table("ipl_analytics.silver.deliveries")

from pyspark.sql.functions import sum, count

player_perf = deliveries.groupBy("batsman").agg(
    sum("runs_off_bat").alias("total_runs"),  # Note: verify if column is runs_off_bat or total_runs
    count("*").alias("balls_faced"),
    sum("extras").alias("extras")
)

In [0]:
fact_player_performance = player_perf.join(
    dim_player.select("player_key", "player_name"),
    player_perf.batsman == dim_player.player_name,
    "left"
).select(
    "player_key",
    "batsman",
    "total_runs",
    "balls_faced",
    "extras"
)

In [0]:
%sql
SHOW TABLES IN ipl_analytics.silver;


In [0]:
display(
    spark.table("ipl_analytics.silver.deliveries")
)

In [0]:
display(
    spark.table("ipl_analytics.silver.deliveries")
         .select("batsman")
         .distinct()
)

### ------------------------------------------------

In [0]:
display(spark.table("ipl_analytics.gold.dim_player"))

In [0]:
display(spark.table("ipl_analytics.gold.dim_team"))

In [0]:
display(spark.table("ipl_analytics.gold.dim_venue"))

In [0]:
display(spark.table("ipl_analytics.gold.fact_match_summary"))

In [0]:
deliveries = spark.table("ipl_analytics.silver.deliveries")

fact_player_performance = (
    deliveries
    .join(
        dim_player,
        deliveries.batsman == dim_player.player_name,
        "left"
    )
    .groupBy(
        "player_key",
        "match_id"
    )
    .agg(
        sum("runs_off_bat").alias("total_runs"),
        count("*").alias("balls_faced"),
        sum("extras").alias("extras"),
        sum(when(col("runs_off_bat") == 4, 1).otherwise(0)).alias("fours"),
        sum(when(col("runs_off_bat") == 6, 1).otherwise(0)).alias("sixes"),
        sum("is_wicket").alias("dismissals")
    )
    .filter(col("player_key").isNotNull())  # <-- Filters out null player_keys
)

In [0]:
(fact_player_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ipl_analytics.gold.fact_player_performance"))

In [0]:
display(spark.table("ipl_analytics.gold.fact_player_performance"))


In [0]:
spark.table("ipl_analytics.gold.fact_player_performance").printSchema()

-----------------------------
### `------------------------------
`

In [0]:
from pyspark.sql.functions import col

fact_player_performance.filter(col("player_key").isNull()).count()

In [0]:
deliveries_df.select("batsman").distinct().show(50, False)

In [0]:
players_df.select("player_name").distinct().show(50, False)